In [ ]:
#| default_exp core

# core

> One `Model` over every runtime, one `Pred` back, and the file moves that follow from a prediction.

`Model(path_or_repo)` picks the runtime from the shape of the name and returns that runtime's own
subclass, the way `rishi.Chat` picks a chat backend. Everything after that is the same call whether
the weights are `.tflite`, `.onnx` or `.mlpackage`.

In [ ]:
#| export
from __future__ import annotations
import json, os, shutil
from importlib import import_module
from pathlib import Path

import numpy as np
from fastcore.all import AttrDict, L, store_attr

from anya.vision import (MEDIA_EXTS, IMG_EXTS, AUD_EXTS, VID_EXTS, AudioPrep, Prep, batch, decode_classify,
                         decode_detect_auto, decode_segment, load_audio, load_image, pool_embed, read_labels,
                         video_frames)

In [ ]:
#| hide
from fastcore.test import *
from tempfile import mkdtemp

## Items

`items` is the only argument shape the rest of the library takes: a file, a folder, a glob, a list,
or an already-loaded image. A folder of 2000 bird photos and one `PIL.Image` go in through the same
door.

In [ ]:
#| export
def is_loaded(o) -> bool:
    'Is `o` already pixels or bytes rather than something to read off disk?'
    return isinstance(o, (np.ndarray, bytes, bytearray)) or (hasattr(o, 'mode') and hasattr(o, 'size'))

def item_src(o) -> str|None:
    'A printable source for `o`, or None when it came in as pixels.'
    return None if is_loaded(o) else str(o)

def items(o,                      # a path, a folder, a glob, an iterable of any of those, or pixels
          types:str='image',      # 'image', 'audio', 'video', 'any', or an iterable of suffixes
          recurse:bool=True,      # walk subdirectories of a folder
          sort:bool=True,         # deterministic order, so a rerun matches the last run
          exclude=None            # paths to skip: a sorted output folder under the folder being read
         ) -> L:
    'Expand `o` into a flat `L` of things a model can be run on.'
    exts = _exts(types)
    if o is None: return L()
    if is_loaded(o): return L([o])
    if isinstance(o, (str, Path)):
        p = Path(o).expanduser()
        if p.is_dir():
            fs = L(p.rglob('*') if recurse else p.glob('*')).filter(lambda f: f.is_file() and f.suffix.lower() in exts)
        elif any(c in str(o) for c in '*?['): fs = L(Path().glob(str(o)))
        else: fs = L([p])
        fs = _drop_excluded(fs, exclude)
        return fs.sorted() if sort else fs
    return L(o).map(lambda x: items(x, types=types, recurse=recurse, sort=sort, exclude=exclude)).concat()

def _drop_excluded(fs, exclude) -> L:
    'Files under any of `exclude` are not items: this is what stops a second run reading its own output.'
    if not exclude: return fs
    ex = L(exclude if isinstance(exclude, (list, tuple, L)) else [exclude]).map(lambda p: str(Path(p).expanduser().absolute()))
    return fs.filter(lambda f: not any(str(Path(f).absolute()).startswith(e + os.sep) for e in ex))

def _exts(types) -> set:
    if types in (None, 'any', 'all'): return MEDIA_EXTS
    if isinstance(types, str): return {'image': IMG_EXTS, 'audio': AUD_EXTS, 'video': VID_EXTS}[types]
    return {e if e.startswith('.') else '.'+e for e in types}

In [ ]:
#| hide
_td = Path(mkdtemp())
(_td/'sub').mkdir()
for f in ['b.jpg', 'a.png', 'notes.txt', 'sub/c.jpeg']: (_td/f).write_bytes(b'x')
test_eq(items(_td).attrgot('name'), ['a.png', 'b.jpg', 'c.jpeg'])     # sorted, recursive, images only
test_eq(len(items(_td, recurse=False)), 2)
test_eq(len(items(_td, types='any')), 3)                             # .txt is not media in any case
test_eq(len(items([_td/'a.png', _td/'b.jpg'])), 2)
test_eq(len(items(np.zeros((4,4,3), np.uint8))), 1)
test_eq(item_src(np.zeros((2,2,3), np.uint8)), None)
test_eq(items(_td, exclude=_td/'sub').attrgot('name'), ['a.png', 'b.jpg'])

## Results

A `Pred` is a dict, so it survives `json.dumps` and crosses a tool call without a serialiser. It
also answers `.label` and `.score` whatever the task was, which is what calling code wants when it
is sorting a folder and does not care that the model was a detector.

In [ ]:
#| export
class Pred(AttrDict):
    'One result: what it ran on, which model, and the task-shaped payload.'
    @property
    def label(self) -> str|None:
        'The single most useful label: top class, biggest object, or largest segment.'
        if self.get('error'): return None
        for k in ('preds', 'objects', 'classes'):
            if self.get(k): return self[k][0]['label']
        return self.get('text')

    @property
    def score(self) -> float|None:
        'Confidence behind `label`.'
        for k in ('preds', 'objects'):
            if self.get(k): return self[k][0]['score']
        if self.get('classes'): return self['classes'][0]['frac']
        return None

    def json(self) -> dict:
        'A JSON-safe copy: arrays are replaced by their shape.'
        return {k: (dict(shape=list(np.shape(v)), dtype=str(np.asarray(v).dtype)) if isinstance(v, np.ndarray) else v)
                for k, v in self.items()}

    def _repr_markdown_(self):
        h = f"**{self.label}**" + (f" ({self.score:.3f})" if self.score is not None else '')
        if self.get('error'): h = f"**error**: {self['error']}"
        rows = [f"- {o['label']} {o['score']:.3f} {o.get('box','')}" for o in (self.get('objects') or [])[:8]]
        rows += [f"- {p['label']} {p['score']:.3f}" for p in (self.get('preds') or [])[1:6]]
        rows += [f"- {c['label']} {c['frac']:.1%}" for c in (self.get('classes') or [])[1:6]]
        return '\n'.join([f"`{self.get('src') or 'array'}` → {h}", ''] + rows)

In [ ]:
#| export
class Preds(L):
    'The results of one run, with the summaries a caller actually asks for.'
    @property
    def labels(self) -> L:
        'The one label per item, in order. `attrgot` would read the dict key, not the property.'
        return L(p.label for p in self)
    @property
    def ok(self) -> Preds: return self.filter(lambda p: not p.get('error'))
    @property
    def failed(self) -> Preds: return self.filter(lambda p: p.get('error'))

    def counts(self) -> dict:
        'How many items got each label, most common first.'
        c = {}
        for l in self.ok.labels: c[l] = c.get(l, 0) + 1
        return dict(sorted(c.items(), key=lambda t: -t[1]))

    def above(self, score:float) -> Preds:
        'Only the predictions the model was at least `score` confident about.'
        return self.ok.filter(lambda p: (p.score or 0) >= score)

    def by_label(self) -> dict:
        'Group into `{label: Preds}`.'
        d = {}
        for p in self.ok: d.setdefault(p.label, Preds()).append(p)
        return d

    def records(self) -> list:
        'JSON-safe dicts, one per item.'
        return [p.json() for p in self]

    def save(self, path) -> Path:
        'Write `records()` to a JSON file.'
        p = Path(path); p.write_text(json.dumps(self.records(), indent=1)); return p

    def _repr_markdown_(self):
        n = len(self)
        head = f"{n} item{'s' if n != 1 else ''}" + (f", {len(self.failed)} failed" if self.failed else '')
        rows = [f"| `{Path(p.get('src') or 'array').name}` | {p.label} | {p.score:.3f} |" if p.score is not None
                else f"| `{Path(p.get('src') or 'array').name}` | {p.label or p.get('error')} | |" for p in self[:20]]
        tbl = ['', '| item | label | score |', '|---|---|---|'] + rows + (['| … | | |'] if n > 20 else [])
        return '\n'.join([head] + tbl)

In [ ]:
#| hide
_ps = Preds([Pred(src='a.jpg', preds=[dict(label='wren', score=0.9, index=1)]),
             Pred(src='b.jpg', preds=[dict(label='wren', score=0.4, index=1)]),
             Pred(src='c.jpg', error='cannot identify image file')])
test_eq(_ps.counts(), {'wren': 2})
test_eq(len(_ps.above(0.5)), 1)
test_eq(len(_ps.failed), 1)
test_eq(list(_ps.by_label()), ['wren'])
test_eq(_ps[0].label, 'wren'); test_eq(_ps[2].label, None)
test_eq(json.loads(json.dumps(_ps.records()))[0]['src'], 'a.jpg')
test_eq(Pred(src='x', vec=np.zeros(4, np.float32)).json()['vec'], dict(shape=[4], dtype='float32'))

## Which runtime

Resolution order is rishi's: an explicit `runtime=`, then a `runtime/` prefix, then the shape of the
id or path. A bare hub repo id decides nothing on its own, so `Model` asks `anya.hub` which file the
repo actually ships rather than guessing.

In [ ]:
#| export
runtimes = {'onnx':   ('anya.onnx',   'OnnxModel'),
            'litert': ('anya.litert', 'LitertModel'),
            'coreml': ('anya.apple',  'CoreMLModel')}

# checked in order, so a suffix wins over a word appearing anywhere in a repo id
_pats = {'litert': ('.tflite', '.lite', 'litert', 'tflite'),
         'onnx':   ('.onnx', '.ort', 'onnx'),
         'coreml': ('.mlpackage', '.mlmodel', '.mlmodelc', 'coreml', 'core-ml')}

def split_runtime(model):
    "Split `'runtime/model'` into `(runtime, model)`; the prefix must name a known runtime, else `(None, model)`."
    if isinstance(model, str) and '/' in model:
        b, m = model.split('/', 1)
        if b in runtimes: return b, m
    return None, model

def infer_runtime(model):
    "Guess a runtime from the shape of a model id or path (`.tflite` vs `.onnx`), else `None`."
    s = str(model or '').lower()
    if not s: return None
    return next((b for b, ps in _pats.items() if any(p in s for p in ps)), None)

def resolve_runtime(model=None, runtime=None, model_path=None):
    "Resolve `(runtime, model)`; `runtime` is `None` when only the hub can say what a repo ships."
    pre, model = split_runtime(model)
    nm = runtime or pre or infer_runtime(model) or infer_runtime(model_path)
    if nm is not None and nm not in runtimes:
        raise ValueError(f"Unknown runtime {nm!r}; known runtimes: {', '.join(runtimes)}.")
    return nm, model

def _runtime_mod(nm):
    "Import the runtime module for `nm`, with an actionable error when its wheel is missing."
    try: return import_module(runtimes[nm][0])
    except ImportError as e:
        raise ImportError(f"The {nm!r} runtime is unavailable ({e}). "
                          f"Install it with: pip install 'anya[{nm}]'") from None

def get_runtime(nm):
    "The `Model` subclass for runtime `nm` (imports the runtime module lazily)."
    return getattr(_runtime_mod(nm), runtimes[nm][1])

In [ ]:
#| hide
test_eq(resolve_runtime('models/yolo11n.onnx'), ('onnx', 'models/yolo11n.onnx'))
test_eq(resolve_runtime('litert-community/bird-classifier')[0], 'litert')
test_eq(resolve_runtime('org/repo'), (None, 'org/repo'))            # only the hub can tell
test_eq(resolve_runtime('onnx/org/repo'), ('onnx', 'org/repo'))     # an explicit prefix
test_eq(resolve_runtime('org/repo', runtime='litert')[0], 'litert')
test_eq(infer_runtime('a.mlpackage'), 'coreml')
test_fail(lambda: resolve_runtime('x', runtime='jax'), contains='Unknown runtime')

## Tasks

The task decides how an output tensor becomes an answer. A runtime infers it from the model's own
output shapes, and `task=` overrides when the guess is wrong.

In [ ]:
#| export
TASKS = ('classify', 'detect', 'segment', 'embed')

# a single vector this wide, with no labels to go with it, is a feature vector rather than classes
EMBED_DIMS = {64, 128, 192, 256, 320, 384, 512, 640, 768, 896, 1024, 1152, 1280, 1408, 1536, 2048, 4096}

def dims(shape) -> list:
    'The concrete dimensions of a declared shape; symbolic axes (a named batch) drop out.'
    return [d for d in (d if isinstance(d, int) and d > 0 else None for d in shape) if d]

def infer_task(shapes,          # output shapes the model declares, in order
               labels=None,     # class names, when the model or the caller supplied them
               names=None       # output tensor names, which often say it outright
              ) -> str:
    'Guess the task from output shapes: four heads is a detector, a grid of classes is a segmenter.'
    # 'logits' is what a classifier and a segmenter both call their output, so it says nothing
    ns = ' '.join(str(n).lower() for n in (names or []))
    for k, t in (('box', 'detect'), ('mask', 'segment'), ('segment', 'segment'),
                 ('embed', 'embed'), ('feature', 'embed'), ('hidden', 'embed')):
        if k in ns: return t
    shapes = L(shapes)
    if len(shapes) >= 3: return 'detect'                   # boxes / classes / scores / count
    s = list(shapes[0]) if len(shapes) else []
    if len(s) > 1 and (s[0] == 1 or not isinstance(s[0], int)): s = s[1:]   # drop the batch axis
    if len(s) >= 3: return 'segment'                       # a class per pixel, C,H,W or H,W,C
    d = dims(s)
    if len(d) == 2:
        a, b = sorted(d)
        return 'detect' if b >= 100 and a <= 512 else 'classify'
    if len(d) == 1 and labels is None and d[0] in EMBED_DIMS: return 'embed'
    return 'classify'

In [ ]:
#| hide
test_eq(infer_task([(1, 1000)]), 'classify')
test_eq(infer_task([(1, 84, 8400)]), 'detect')
test_eq(infer_task([(1, 25200, 85)]), 'detect')
test_eq(infer_task([(1, 21, 224, 224)]), 'segment')
test_eq(infer_task([(1,4),(1,1),(1,1),(1,)]), 'detect')
test_eq(infer_task([(1, 512)], names=['embedding']), 'embed')
test_eq(infer_task([('batch', 128)]), 'embed')            # a symbolic batch axis, and a round dimension
test_eq(infer_task([(1, 4)]), 'classify')                 # four of anything is classes, not features
test_eq(infer_task([(1, 3)], labels=['a','b','c']), 'classify')
test_eq(dims([1, 'batch', 0, 640]), [1, 640])
# what the three commonest optimum exports declare: every axis symbolic, the name the only clue
test_eq(infer_task([('batch_size', 1000)], names=['logits']), 'classify')
test_eq(infer_task([('batch_size', 'num_labels', 'height', 'width')], names=['logits']), 'segment')
test_eq(infer_task([('batch_size', 257, 384)], names=['last_hidden_state']), 'embed')

### From a model's own signature

`prep_from_spec` reads an input tensor's shape and dtype and returns the `Prep` that fits it: which
axis is channels, what size to resize to, whether the model wants integers. What a signature cannot
say is how the training data was normalised, so `norm` names that, and `anya.hub` fills it in from a
repo's `preprocessor_config.json` when there is one.

In [ ]:
#| export
from anya.vision import BILINEAR, IMAGENET

NORMS = {'01':       ((0., 0., 0.), (1., 1., 1.)),        # pixels in 0..1
         'imagenet': IMAGENET,                            # torchvision and timm exports
         'signed':   ((.5, .5, .5), (.5, .5, .5)),        # pixels in -1..1, most TF exports
         'none':     ((0., 0., 0.), (1., 1., 1.))}        # raw 0..255

CHANNELS = (1, 3, 4)

def chan_axis(shape) -> int:
    'Which axis of an image input holds the channels: 1 for NCHW, -1 for NHWC.'
    # optimum exports every axis symbolic and names them, so the name is all there is to read
    named = lambda i: (nm := str(shape[i]).lower()) == 'c' or 'chan' in nm
    if named(1): return 1
    if named(-1): return -1
    return 1 if (shape[1] in CHANNELS and shape[-1] not in CHANNELS) else -1

def prep_from_spec(shape,                 # the input tensor shape the model declares
                   dtype:str='float32',   # its dtype
                   norm='01',             # a NORMS name, or an explicit (mean, std)
                   size:tuple=None,       # size for the axes the shape leaves symbolic
                   resize:str=None,       # 'stretch', 'letterbox', 'center_crop'
                   crop_pct:float=None,   # fraction of the short side kept by 'center_crop'
                   resample:int=None,     # PIL resample filter; bilinear unless the config says otherwise
                   quant:tuple=None,      # (scale, zero_point) for a quantised input
                   task:str=None,         # only used to pick a default size and resize mode
                   layout:str=None,       # override the channel-axis guess
                   bgr:bool=False
                  ):
    'The `Prep` (or `AudioPrep`) that one input signature calls for.'
    s = list(shape)
    if len(s) <= 2:                                        # (batch, samples) or (samples,): a waveform
        n = dims(s)
        return AudioPrep(samples=(n[-1] if n and n[-1] > 16 else None), dtype=dtype,
                         layout='nt' if len(s) == 2 else 't')
    d = [x if isinstance(x, int) and x > 0 else None for x in s]
    lay = layout or ('nchw' if chan_axis(s) == 1 else 'nhwc')
    hw = (d[2], d[3]) if lay == 'nchw' else (d[1], d[2])
    dflt = 640 if task == 'detect' else 224
    # a size the graph states outright wins: it is the only one the graph will accept
    size = tuple(a or b or dflt for a, b in zip(hw, tuple(size) if size else (None, None)))
    mean, std = NORMS[norm] if isinstance(norm, str) else norm
    scale = 1.0 if norm == 'none' else 1/255
    return Prep(size=size, layout=lay, dtype=dtype, scale=scale, mean=mean, std=std, crop_pct=crop_pct,
                resample=resample or BILINEAR, quant=quant, bgr=bgr,
                resize=resize or ('letterbox' if task == 'detect' else 'stretch'))

In [ ]:
#| hide
test_eq(prep_from_spec([1,3,224,224]).layout, 'nchw')
test_eq(prep_from_spec([1,224,224,3]).layout, 'nhwc')
test_eq(prep_from_spec([1,3,None,None], task='detect').size, (640, 640))
test_eq(prep_from_spec([1,3,None,None], task='detect').resize, 'letterbox')
test_close(prep_from_spec([1,3,8,8], norm='imagenet').mean[0], 0.485)
test_eq(prep_from_spec([1,8,8,3], norm='none').scale, 1.0)
test_eq(prep_from_spec([1, 15600]).samples, 15600)        # a waveform, not a picture
test_eq(prep_from_spec([1,3,224,224], size=(128,128)).size, (224,224))   # the graph's size, not the config's
test_eq(prep_from_spec(['b','c','h','w'], size=(256,256)).size, (256,256))
test_eq(chan_axis(['batch_size','num_channels','height','width']), 1)    # nothing but the names to go on
test_eq(chan_axis([1,224,224,3]), -1)
test_eq(prep_from_spec(['batch_size','num_channels','height','width']).layout, 'nchw')

## Model

Subclasses supply three things: `_load` (open the file), `_infer` (run one batch), and `spec`
(input and output shapes). Everything else, including preprocessing, batching, decoding and the
error handling that keeps one unreadable JPEG from ending a 2000-file run, lives here.

In [ ]:
#| export
class Model:
    "Runtime-agnostic model: `Model(name)` dispatches to the onnx/litert/coreml subclass."
    _runtime = None

    def __new__(cls, model=None, *, runtime=None, model_path=None, **kw):
        # `__new__` only picks the class: Python then calls the subclass `__init__` with the original
        # arguments, so anything resolved here would be thrown away. `model_file` re-resolves, cached.
        if cls is not Model: return super().__new__(cls)
        nm, m = resolve_runtime(model, runtime, model_path)
        if nm is None:
            from anya.hub import resolve_model
            nm = resolve_model(m, file=kw.get('file'), revision=kw.get('revision'))[0]
        return super().__new__(get_runtime(nm))

    def _setup(self, model=None, model_path=None, task=None, labels=None, prep=None,
               topk:int=5, conf:float=0.25, iou:float=0.45, meta:dict=None):
        'Shared init tail: store the knobs every task shares and normalise the labels.'
        _, model = split_runtime(model)
        store_attr('model,model_path,topk,conf,iou', self)
        self.labels = read_labels(labels)
        self.meta = dict(meta or {})
        self._task, self._prep = task, prep
        return model

    @property
    def runtime(self) -> str: return self._runtime
    @property
    def task(self) -> str: return self._task
    @property
    def prep(self) -> Prep: return self._prep
    @property
    def name(self) -> str: return str(self.model or self.model_path)

    def __repr__(self):
        n = f', {len(self.labels)} labels' if self.labels else ''
        return f'{type(self).__name__}({Path(self.name).name}, runtime={self.runtime}, task={self.task}{n})'

    def _infer(self, x) -> list:
        'Run one preprocessed batch; returns the output arrays in declared order.'
        raise NotImplementedError

    @property
    def modality(self) -> str:
        "'audio' when the model wants a waveform, else 'image'."
        return 'audio' if isinstance(self.prep, AudioPrep) else 'image'

    def _load_one(self, o):
        'Read one item into the array this model takes.'
        if is_loaded(o) and not isinstance(o, (bytes, bytearray)) and self.modality == 'audio': return np.asarray(o, np.float32)
        return load_audio(o, self.prep.sr)[0] if self.modality == 'audio' else load_image(o)

    def decode(self, outs, meta:dict, src=None, **kw) -> Pred:
        'Turn raw output arrays into a `Pred` for this model\'s task.'
        d = dict(src=src, task=self.task, model=self.name)
        t, o0 = self.task, np.asarray(outs[0])
        if t == 'classify': return Pred(d, preds=decode_classify(o0, self.labels, kw.get('topk', self.topk)))
        if t == 'detect': return Pred(d, objects=decode_detect_auto(outs, self.labels, meta=meta,
                                                                   conf=kw.get('conf', self.conf), iou=kw.get('iou', self.iou)))
        if t == 'segment': return Pred(d, **decode_segment(o0, self.labels, meta=meta))
        if t == 'embed': return Pred(d, vec=pool_embed(o0))
        return Pred(d, raw=[np.asarray(o) for o in outs])

    def predict(self, o, **kw) -> Pred:
        'Run the model on one item.'
        a = self._load_one(o)
        x, ms = batch([a], self.prep)
        return self.decode(self._infer(x), ms[0], item_src(o), **kw)

    def predict_all(self, o,                    # anything `items` accepts
                    bs:int=None,                # batch size; the model's own limit caps it
                    on_error:str='skip',        # 'skip' records the error and carries on, 'raise' stops
                    types:str=None,             # which files a folder contributes; defaults to the model's modality
                    exclude=None,               # paths to skip, such as the folder the results are being written to
                    **kw
                   ) -> Preds:
        'Run the model over a folder, a list or a glob.'
        xs = items(o, types=types or self.modality, exclude=exclude)
        bs = min(bs or self.max_bs, self.max_bs)
        out = Preds()
        for i in range(0, len(xs), bs):
            chunk = xs[i:i+bs]
            res, loaded = [None]*len(chunk), []
            for k, x in enumerate(chunk):
                try: loaded.append((k, x, self._load_one(x)))       # one unreadable file, not one lost batch
                except Exception as e:
                    if on_error == 'raise': raise
                    res[k] = self._on_error(x, e)
            if loaded:
                try:
                    arr, ms = batch([a for _, _, a in loaded], self.prep)
                    outs = self._infer(arr)
                    for j, (k, x, _) in enumerate(loaded):
                        res[k] = self.decode([np.asarray(o)[j:j+1] for o in outs], ms[j], item_src(x), **kw)
                except Exception as e:
                    if on_error == 'raise': raise
                    for k, x, _ in loaded: res[k] = self._on_error(x, e)
            out += [r for r in res if r is not None]
        return out

    def predict_video(self, path,                 # one video file
                      every:float=1.0,            # seconds between sampled frames
                      max_frames:int=None,
                      **kw
                     ) -> Preds:
        'Sample frames out of a video and run the model on each; every `Pred` carries its timestamp.'
        out = Preds()
        for t, f in video_frames(path, every=every, max_frames=max_frames):
            p = self.predict(f, **kw)
            p['src'], p['t'] = f'{path}#t={t:.2f}', round(t, 3)
            out.append(p)
        return out

    def _on_error(self, x, e) -> Pred:
        return Pred(src=item_src(x), task=self.task, model=self.name, error=f'{type(e).__name__}: {e}')

    @property
    def max_bs(self) -> int:
        'Largest batch the loaded graph accepts; 1 unless the runtime says otherwise.'
        return getattr(self, '_max_bs', 1)

    def __call__(self, o, **kw):
        'One item in, one `Pred` out; a folder or list in, `Preds` out.'
        if is_loaded(o) or (isinstance(o, (str, Path)) and not Path(str(o)).is_dir() and not any(c in str(o) for c in '*?[')):
            return self.predict(o, **kw)
        return self.predict_all(o, **kw)

    def close(self):
        'Release the graph. A closed model raises if called again.'
        self._sess = None
    def __enter__(self): return self
    def __exit__(self, *a): self.close()

In [ ]:
#| export
def model_file(model=None,       # a path or a hub repo id
               model_path=None,  # an explicit file, which wins
               file:str=None,    # which file inside a repo, when the repo ships several
               revision:str=None
              ) -> Path:
    'The local file a runtime should open, downloading it from the hub the first time if needed.'
    if model_path: return Path(model_path).expanduser()
    p = Path(str(model)).expanduser()
    if p.exists(): return p
    from anya.hub import resolve_model
    return Path(resolve_model(str(model), file=file, revision=revision)[1])

In [ ]:
#| export
def sidecar_labels(path) -> Path|None:
    'A labels file beside the weights or up at the repo root, which is how most exports ship class names.'
    p = Path(path)
    cs = [p.with_suffix('.txt')] + [d/n for d in list(p.parents)[:3]
                                    for n in ('labels.txt', 'classes.txt', 'labelmap.txt', 'config.json')]
    return next((c for c in cs if c.exists()), None)

def with_hub_defaults(path,           # the local weights file
                      labels=None,    # class names, if the caller has them
                      **kw            # anything `prep_from_spec` takes; None means unset
                     ) -> tuple:
    'Returns `(labels, prep keywords)`, filling whatever the caller left unset from the configs beside `path`.'
    kw = {k: v for k, v in kw.items() if v is not None}
    try: from anya.hub import model_config, prep_kwargs
    except ImportError: return labels, kw
    c = model_config(path)
    return (labels if labels is not None else read_labels(c.config), {**prep_kwargs(c.preprocessor), **kw})

### Loading once

An agent calling `classify` on three folders should load the weights once. `load_model` caches by
the arguments that change the graph, so repeated tool calls reuse one session.

In [ ]:
#| export
_models = {}

def load_model(model=None, **kw) -> Model:
    'A cached `Model`, expanding an alias from the registry; the same arguments return the same object.'
    try:
        from anya.hub import resolve_alias
        if (a := resolve_alias(model)): model, kw = a['model'], {**{x: y for x, y in a.items() if x != 'model'}, **kw}
    except ImportError: pass
    k = (str(model), tuple(sorted((a, str(b)) for a, b in kw.items())))
    if k not in _models: _models[k] = Model(model, **kw)
    return _models[k]

def loaded_models() -> L:
    'What `load_model` is currently holding open.'
    return L(_models.values())

def clear_models():
    'Close and drop every cached model.'
    for m in _models.values():
        try: m.close()
        except Exception: pass
    _models.clear()

## Arranging files

The point of classifying 2000 photos is usually to put them somewhere. `arrange` turns `Preds` into
a plan and, only when asked, carries it out. `dry_run=True` is the default because an agent holding
this tool should show its working before it moves anyone's files.

In [ ]:
#| export
def safe_name(s:str, mx:int=60) -> str:
    'A label as a directory name: no separators, no surprises.'
    out = ''.join(c if (c.isalnum() or c in ' -_') else '_' for c in str(s)).strip().replace(' ', '_')
    return (out[:mx] or 'unlabelled').strip('._')

def arrange(preds:Preds,             # what a model returned
            dest,                    # directory to build under
            how:str='copy',          # 'copy', 'move', or 'link' (a hard link, so nothing is duplicated)
            min_score:float=0.0,     # anything less confident goes to `other`
            other:str='unsorted',    # folder for low-confidence and failed items
            dry_run:bool=True,       # print the plan and change nothing
            depth:int=1              # 1 = dest/label/file, 2 = dest/label/sub-label/file for hierarchical names
           ) -> AttrDict:
    'Sort the files behind `preds` into `dest/<label>/`, returning the plan and what was done.'
    dest = Path(dest).expanduser()
    plan, seen = [], set()
    for p in preds:
        src = p.get('src')
        if not src: continue
        lab = p.label if (not p.get('error') and (p.score or 0) >= min_score) else None
        parts = [safe_name(x) for x in str(lab).split('/')[:depth]] if lab else [safe_name(other)]
        d = dest.joinpath(*parts)/Path(src).name
        n = 1
        while str(d) in seen or (d.exists() and not dry_run and Path(src).resolve() != d.resolve()):
            d = d.with_name(f'{Path(src).stem}_{n}{Path(src).suffix}'); n += 1
        seen.add(str(d))
        plan.append(dict(src=str(src), dst=str(d), label=lab))
    done = 0
    if not dry_run:
        for a in plan:
            Path(a['dst']).parent.mkdir(parents=True, exist_ok=True)
            _place(a['src'], a['dst'], how); done += 1
    return AttrDict(dest=str(dest), how=how, dry_run=dry_run, n=len(plan), moved=done,
                    labels=sorted({a['label'] for a in plan if a['label']}), plan=plan)

def _place(src, dst, how):
    if how == 'move': shutil.move(src, dst)
    elif how == 'link':
        try: os.link(src, dst)
        except OSError: shutil.copy2(src, dst)     # a hard link cannot cross a filesystem
    else: shutil.copy2(src, dst)

In [ ]:
#| hide
_ad = Path(mkdtemp())
for f in ['x.jpg', 'y.jpg', 'z.jpg']: (_ad/f).write_bytes(b'x')
_pr = Preds([Pred(src=str(_ad/'x.jpg'), preds=[dict(label='Superb Fairywren', score=0.9, index=0)]),
             Pred(src=str(_ad/'y.jpg'), preds=[dict(label='Superb Fairywren', score=0.2, index=0)]),
             Pred(src=str(_ad/'z.jpg'), error='broken')])
_r = arrange(_pr, _ad/'out', min_score=0.5)
test_eq(_r.dry_run, True); test_eq(_r.moved, 0); test_eq((_ad/'out').exists(), False)
test_eq([Path(a['dst']).parent.name for a in _r.plan], ['Superb_Fairywren', 'unsorted', 'unsorted'])
_r = arrange(_pr, _ad/'out', how='copy', min_score=0.5, dry_run=False)
test_eq(_r.moved, 3)
test_eq((_ad/'out'/'Superb_Fairywren'/'x.jpg').exists(), True)
test_eq((_ad/'out'/'unsorted'/'z.jpg').exists(), True)
test_eq(safe_name('Malurus cyaneus / male'), 'Malurus_cyaneus___male')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()